# Random Forests — Random Forests (2001)

_Papers · Classical ML_

**Grow many deep, deliberately decorrelated decision trees — each on a bootstrap sample and splitting on a random subset of features — then let them vote; the average is far more stable than any one tree and does not overfit as you add trees.**

---

This notebook is a practice scaffold for the **Random Forests — Random Forests (2001)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

### Step 1 — Worked examples: the two ideas behind a forest

A random forest rests on two facts. First, **bagging** turns many tree predictions into one by a *majority vote* (Definition 1.1). Second, the **variance of an average** of $B$ correlated predictors is $\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$ — so as $B$ grows the second term vanishes and the variance falls toward the correlation floor $\rho\sigma^2$. That floor is exactly why a forest works to **decorrelate** its trees: lowering $\rho$ lowers the floor.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# ---- worked example 1: bagging majority vote (Definition 1.1) ----
votes = np.array([1, 1, 0, 1, 0])                 # 5 trees vote on one point
tally = np.bincount(votes)
majority = int(tally.argmax())
print("votes:", votes.tolist(), "-> tally", tally.tolist(), "-> majority", majority)   # class 1 (3 vs 2)

# ---- worked example 2: variance of an average vs correlation (the derivation) ----
# Var = rho*sigma^2 + (1-rho)/B * sigma^2 ; sigma^2 = 1, B = 100
for rho in (0.9, 0.6, 0.3):
    var = rho + (1 - rho) / 100
    print(f"rho={rho}:  Var/sigma^2 = {var:.4f}")   # 0.9010, 0.6040, 0.3070

### Step 2 — Build the toy data and a single decision tree

We make an XOR-like 2-D set: two Gaussian clusters per class, so no straight line separates them and a tree must carve out several regions. The `Tree` class grows to full depth with no pruning, choosing each split by the **weighted Gini impurity**. The key forest hook is `best_split`: if `max_features` is set, it considers only a *random subset* of features at each node — the per-split randomness (Forest-RI) that decorrelates the trees.

In [ ]:
# ---- toy 2-D data: 2 gaussian clusters per class (XOR-like, nonlinear) ----
def make_data(n, seed):
    r = np.random.RandomState(seed)
    X0 = np.vstack([r.randn(n//2, 2)*0.7 + [-1, -1], r.randn(n//2, 2)*0.7 + [2, 2]])
    X1 = np.vstack([r.randn(n//2, 2)*0.7 + [-1, 2], r.randn(n//2, 2)*0.7 + [2, -1]])
    return np.vstack([X0, X1]), np.array([0]*len(X0) + [1]*len(X1))

Xtr, ytr = make_data(200, 1)
Xte, yte = make_data(400, 2)

# ---------------- a single decision tree, grown to full depth ----------------
class Tree:
    def __init__(self, max_features=None, rng=None):
        self.max_features = max_features
        self.rng = rng

    def gini(self, y):
        if len(y) == 0:
            return 0.0
        p = np.bincount(y, minlength=2) / len(y)
        return 1.0 - (p**2).sum()

    def best_split(self, X, y):
        n, d = X.shape
        feats = np.arange(d)
        if self.max_features is not None and self.max_features < d:     # RANDOM SUBSET per node
            feats = self.rng.choice(d, self.max_features, replace=False)
        best, bestg = None, 1e9
        for f in feats:
            vals = np.unique(X[:, f])
            for i in range(len(vals) - 1):
                thr = (vals[i] + vals[i+1]) / 2
                L = X[:, f] <= thr
                if L.sum() == 0 or (~L).sum() == 0:
                    continue
                g = (L.sum()*self.gini(y[L]) + (~L).sum()*self.gini(y[~L])) / n   # weighted Gini
                if g < bestg:
                    bestg, best = g, (f, thr)
        return best

    def fit(self, X, y):
        self.leaf = None
        if len(np.unique(y)) == 1:                                       # grow to max size, no pruning
            self.leaf = int(y[0])
            return self
        sp = self.best_split(X, y)
        if sp is None:
            self.leaf = int(np.bincount(y, minlength=2).argmax())
            return self
        self.f, self.thr = sp
        L = X[:, self.f] <= self.thr
        self.left = Tree(self.max_features, self.rng).fit(X[L], y[L])
        self.right = Tree(self.max_features, self.rng).fit(X[~L], y[~L])
        return self

    def predict_one(self, x):
        if self.leaf is not None:
            return self.leaf
        return (self.left if x[self.f] <= self.thr else self.right).predict_one(x)

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])

### Step 3 — Assemble the forest: bootstrap, vote, and OOB scoring

The `Forest` grows `n_trees` trees, each on a **bootstrap sample** (drawn with replacement), and predicts by **majority vote**. Each bootstrap leaves out roughly a third of the rows — its **out-of-bag** set — and `oob_score` classifies every point using only the trees that did *not* train on it, giving a built-in, almost-free estimate of generalization error (Section 3.1). We fit our forest with `max_features=1` (one random feature per split, maximally decorrelated).

In [ ]:
class Forest:
    def __init__(self, n_trees=50, max_features=1, seed=0):
        self.n_trees = n_trees
        self.max_features = max_features
        self.rng = np.random.RandomState(seed)

    def fit(self, X, y):
        n = len(X)
        self.trees, self.oob_idx = [], []
        self.X, self.y = X, y
        for _ in range(self.n_trees):
            idx = self.rng.randint(0, n, n)                              # BOOTSTRAP (with replacement)
            oob = np.setdiff1d(np.arange(n), np.unique(idx))             # left-out ~1/3
            self.trees.append(Tree(self.max_features, self.rng).fit(X[idx], y[idx]))
            self.oob_idx.append(oob)
        return self

    def predict(self, X):
        P = np.array([t.predict(X) for t in self.trees])                # (B, n)
        return (P.mean(0) >= 0.5).astype(int)                           # MAJORITY VOTE

    def oob_score(self):
        n = len(self.X)
        votes = np.zeros((n, 2))
        for t, oob in zip(self.trees, self.oob_idx):
            for i, p in zip(oob, t.predict(self.X[oob])):
                votes[i, p] += 1
        seen = votes.sum(1) > 0
        return (votes.argmax(1)[seen] == self.y[seen]).mean()

acc = lambda m, X, y: (m.predict(X) == y).mean()

### Step 4 — Compare, measure variance reduction, and ablate tree count

Now we put the pieces to work. First, a single deep tree vs our forest vs sklearn's `RandomForestClassifier` on held-out test data — and our OOB estimate should track sklearn's. Second, we refit on 20 freshly resampled training sets: the forest's run-to-run accuracy **std** should be roughly half the single tree's, the variance reduction in action. Finally, an **ablation** sweeping the number of trees shows accuracy and OOB climbing then plateauing as trees accumulate.

In [ ]:
single = Tree(max_features=None, rng=np.random.RandomState(0)).fit(Xtr, ytr)  # one deep tree
forest = Forest(n_trees=50, max_features=1, seed=42).fit(Xtr, ytr)            # our forest, F=1
sk = RandomForestClassifier(n_estimators=50, max_features=1, bootstrap=True,
                            oob_score=True, random_state=42).fit(Xtr, ytr)    # sklearn

print("\nTEST ACC  single tree:", round(acc(single, Xte, yte), 3),
      " | our forest:", round(acc(forest, Xte, yte), 3),
      " | sklearn RF:", round(acc(sk, Xte, yte), 3))     # ~0.935 | ~0.959 | ~0.960
print("OOB       ours:", round(forest.oob_score(), 3),
      " | sklearn:", round(sk.oob_score_, 3))            # ~0.96  | ~0.955  -> our forest TRACKS sklearn

# ---- variance reduction: 20 resampled train sets, single tree vs forest ----
st_acc, ft_acc = [], []
for s in range(20):
    Xs, ys = make_data(200, 100 + s)
    st_acc.append(acc(Tree(max_features=None, rng=np.random.RandomState(s)).fit(Xs, ys), Xte, yte))
    ft_acc.append(acc(Forest(n_trees=25, max_features=1, seed=s).fit(Xs, ys), Xte, yte))
print("\nacross 20 train sets  single-tree std:", round(np.std(st_acc), 4),
      " forest std:", round(np.std(ft_acc), 4))          # ~0.0107 vs ~0.0059 (forest ~half)

# ---- ablation: accuracy + OOB vs number of trees ----
print("\nablation (F=1):")
for nt in (1, 2, 5, 10, 25, 50):
    f = Forest(n_trees=nt, max_features=1, seed=7).fit(Xtr, ytr)
    print(f"  trees={nt:2d}  test_acc={acc(f, Xte, yte):.3f}  oob={f.oob_score():.3f}")

## Visualize the data & results

_On a toy 2-D set, does a random forest built from scratch (bootstrap + a random feature subset per split + majority vote) beat a single deep tree and reduce its run-to-run variance — and does it track sklearn's RandomForestClassifier?_

### Step 1 — Re-establish the data and forest in this cell

This visualization section stands on its own, so we re-import and re-declare the same `make_data`, `Tree`, and `Forest` here (identical logic to the reference implementation, just self-contained). Re-stating them means you can run this section without scrolling back up. We then regenerate the same train/test split.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier

def make_data(n, seed):
    r = np.random.RandomState(seed)
    X0 = np.vstack([r.randn(n//2, 2)*0.7 + [-1, -1], r.randn(n//2, 2)*0.7 + [2, 2]])
    X1 = np.vstack([r.randn(n//2, 2)*0.7 + [-1, 2], r.randn(n//2, 2)*0.7 + [2, -1]])
    return np.vstack([X0, X1]), np.array([0]*len(X0) + [1]*len(X1))

class Tree:
    def __init__(self, mf=None, rng=None):
        self.mf, self.rng = mf, rng
    def gini(self, y):
        if len(y) == 0:
            return 0.0
        p = np.bincount(y, minlength=2) / len(y)
        return 1.0 - (p**2).sum()
    def split(self, X, y):
        n, d = X.shape
        feats = np.arange(d)
        if self.mf is not None and self.mf < d:
            feats = self.rng.choice(d, self.mf, replace=False)
        best, bg = None, 1e9
        for f in feats:
            v = np.unique(X[:, f])
            for i in range(len(v) - 1):
                t = (v[i] + v[i+1]) / 2
                L = X[:, f] <= t
                if L.sum() == 0 or (~L).sum() == 0:
                    continue
                g = (L.sum()*self.gini(y[L]) + (~L).sum()*self.gini(y[~L])) / n
                if g < bg:
                    bg, best = g, (f, t)
        return best
    def fit(self, X, y):
        self.leaf = None
        if len(np.unique(y)) == 1:
            self.leaf = int(y[0])
            return self
        sp = self.split(X, y)
        if sp is None:
            self.leaf = int(np.bincount(y, minlength=2).argmax())
            return self
        self.f, self.t = sp
        L = X[:, self.f] <= self.t
        self.l = Tree(self.mf, self.rng).fit(X[L], y[L])
        self.r = Tree(self.mf, self.rng).fit(X[~L], y[~L])
        return self
    def one(self, x):
        return self.leaf if self.leaf is not None else (self.l if x[self.f] <= self.t else self.r).one(x)
    def predict(self, X):
        return np.array([self.one(x) for x in X])

class Forest:
    def __init__(self, B=50, mf=1, seed=0):
        self.B, self.mf, self.rng = B, mf, np.random.RandomState(seed)
    def fit(self, X, y):
        n = len(X)
        self.trees, self.oob = [], []
        self.X, self.y = X, y
        for _ in range(self.B):
            idx = self.rng.randint(0, n, n)
            self.oob.append(np.setdiff1d(np.arange(n), np.unique(idx)))
            self.trees.append(Tree(self.mf, self.rng).fit(X[idx], y[idx]))
        return self
    def predict(self, X):
        return (np.array([t.predict(X) for t in self.trees]).mean(0) >= 0.5).astype(int)
    def oob_score(self):
        n = len(self.X)
        v = np.zeros((n, 2))
        for t, o in zip(self.trees, self.oob):
            for i, p in zip(o, t.predict(self.X[o])):
                v[i, p] += 1
        s = v.sum(1) > 0
        return (v.argmax(1)[s] == self.y[s]).mean()

acc = lambda m, X, y: (m.predict(X) == y).mean()
Xtr, ytr = make_data(200, 1)
Xte, yte = make_data(400, 2)

### Step 2 — Print the four headline results

With the machinery in place we read off the story in four lines: the **tree-count ablation** (accuracy rising as trees are added), the lone **single tree** for reference, the **variance comparison** (forest std vs single-tree std across 20 resamples), and finally **our forest vs sklearn** on both test accuracy and OOB. Together these reproduce the paper's direction — a big single-tree-to-forest gap and an OOB estimate that tracks test accuracy — on our small toy run.

In [ ]:
print("ablation:", [round(acc(Forest(nt, 1, 7).fit(Xtr, ytr), Xte, yte), 3) for nt in (1, 2, 5, 10, 25, 50)])
print("single tree:", round(acc(Tree(None, np.random.RandomState(0)).fit(Xtr, ytr), Xte, yte), 3))

st = [acc(Tree(None, np.random.RandomState(s)).fit(*make_data(200, 100 + s)), Xte, yte) for s in range(20)]
ft = [acc(Forest(25, 1, s).fit(*make_data(200, 100 + s)), Xte, yte) for s in range(20)]
print("std single:", round(np.std(st), 4), "std forest:", round(np.std(ft), 4))

f = Forest(50, 1, 42).fit(Xtr, ytr)
sk = RandomForestClassifier(n_estimators=50, max_features=1, oob_score=True, random_state=42).fit(Xtr, ytr)
print("ours test/oob:", round(acc(f, Xte, yte), 3), round(f.oob_score(), 3),
      "| sklearn:", round(acc(sk, Xte, yte), 3), round(sk.oob_score_, 3))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A forest of 7 trees votes on one point: $[0,1,1,0,1,1,1]$ for a 2-class problem. What does the forest predict, and what is the vote-fraction (margin ingredient) for the winning class?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Tally: class 0 gets 2 votes, class 1 gets 5. — _Majority vote (Definition 1.1) picks the most-voted class._
- Winner is class 1. — _5 > 2._
- Vote-fraction for class 1 = $5/7 \approx 0.714$. — _The forest's confidence; in the two-class margin it is $0.714 - 0.286 = 0.428 > 0$, so a correct, confident vote if the truth is class 1._

**Answer:** Class 1, with a $5/7\approx0.714$ vote-fraction (two-class margin $\approx 0.43$).

</details>

**Problem 2.** Variance of an average. 100 trees each have error variance $\sigma^2=1$. Compute the variance of their average when the pairwise correlation is (a) $\rho=0.9$ and (b) $\rho=0.3$, and say what this implies for forest design.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use $\operatorname{Var}=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2$ with $B=100$, $\sigma^2=1$. — _The variance-of-an-average identity (derivation)._
- (a) $0.9+0.1/100 = 0.901$. — _High correlation: averaging barely helps; the $\rho$ floor dominates._
- (b) $0.3+0.7/100 = 0.307$. — _Lower correlation lets averaging cut variance to about a third._

**Answer:** 0.901 vs 0.307. Same trees, same count — only correlation changed. So the lever in forest design is DECORRELATION: Forest-RI's random per-split feature subset pushes $\rho$ down (Theorem 2.3's $\bar\rho$).

</details>

**Problem 3.** ABLATION (1 tree vs forest). On the toy 2-D set we fit a single deep tree and forests of growing size ($F=1$). Predict how test accuracy and stability change as trees are added, and what OOB error should track.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- One deep tree is high-variance: accurate on its resample but it wobbles a lot across train sets. — _A single split near the root can reshuffle the whole tree (the problem §1.1 motivates)._
- Adding bootstrapped, feature-randomized trees and voting averages away that variance, so accuracy rises then plateaus and run-to-run spread shrinks. — _Variance of the average $\to\rho\sigma^2$ as $B$ grows (derivation); decorrelation lowers the floor._
- OOB error should sit close to test error once enough trees accumulate. — _OOB votes use only the trees that omitted each point — an unbiased generalization estimate (§3.1)._

**Answer:** Accuracy climbs from the single tree (~0.92 in our run) to ~0.96 and flattens; the forest's across-runs std is roughly half the single tree's; OOB tracks test accuracy. This reproduces the paper's direction (Table 2's big single-tree-vs-forest gap) — our small run, not the paper's numbers.

</details>